# PAAL Posture Classification Pipeline

**One-click posture inference on OAK-D pig data.**

This notebook runs a trained MobileNetV2 model on IR images to classify pig postures (standing / sitting / lying). It produces:
- `predictions.csv` — per-frame predictions with confidence scores
- `posture_heatmap.png` — timeline heatmap (pig ID x time)

**How to use:**
1. Upload your model file (`posture3_ir_best.pth`) to Colab or Google Drive
2. Upload or mount your OAK-D data folder (timestamp folders like `20260211-09-17-49/`)
3. Set the two paths in the config cell below
4. Run All Cells

## 0. Install dependencies

In [ ]:
!pip install -q torch torchvision opencv-python-headless matplotlib numpy

## 1. Mount Google Drive (optional)

If your data and model are on Google Drive, run this cell. Otherwise skip it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

**Change these two paths to match your setup:**

In [ ]:
# ============================================================
# CHANGE THESE PATHS
# ============================================================

# Path to the trained model file
MODEL_PATH = "/content/drive/MyDrive/PAAL/posture3_ir_best.pth"

# Path to OAK-D data folder (contains timestamp subfolders like 20260211-09-17-49/)
DATA_DIR = "/content/drive/MyDrive/PAAL/Fed_pig/Pictures_OAk"

# Output directory (results saved here)
OUTPUT_DIR = "/content/paal_outputs"

# ============================================================
# DON'T CHANGE BELOW THIS LINE
# ============================================================

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.exists(MODEL_PATH), f"Model not found: {MODEL_PATH}"
assert os.path.isdir(DATA_DIR), f"Data folder not found: {DATA_DIR}"
print(f"Model: {MODEL_PATH}")
print(f"Data:  {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")

## 3. Pipeline Code

All the inference logic bundled into one cell. No external files needed.

In [ ]:
import csv
import re
from collections import Counter, defaultdict
from datetime import datetime, timedelta

import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# ── Constants ──────────────────────────────────────────────────

IMG_SIZE = 224
TOF_W, TOF_H = 640, 480
DEPTH_THRESHOLD = 1463
BAR_MARGIN = 50
CROP_TOF = (120, 30, 500, 480)  # (x_left, y_top, x_right, y_bottom)

POSTURE3_CLASSES = {0: "standing", 1: "sitting", 2: "lying"}

FILENAME_RE = re.compile(
    r"^pig(\d+)_(depth_vis|depth|ir_vis|ir|rgb_aligned|rgb)_"
    r"(\d{8}-\d{2}-\d{2}-\d{2})(_cropped)?\.(jpg|raw)$"
)
FOLDER_RE = re.compile(r"^\d{8}-\d{2}-\d{2}-\d{2}$")


# ── Model ────────────────────────────────────────────────────

class SingleModalModel(nn.Module):
    def __init__(self, in_channels=3, num_classes=3, pretrained=False, backbone="mobilenet_v2"):
        super().__init__()
        self.backbone_name = backbone
        if backbone == "mobilenet_v2":
            weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
            net = models.mobilenet_v2(weights=weights)
            if in_channels != 3:
                old = net.features[0][0]
                new_conv = nn.Conv2d(in_channels, old.out_channels,
                    kernel_size=old.kernel_size, stride=old.stride,
                    padding=old.padding, bias=old.bias is not None)
                net.features[0][0] = new_conv
            self.features = net.features
            last_ch = net.last_channel
        else:
            raise ValueError(f"This notebook only supports mobilenet_v2, got: {backbone}")
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(last_ch, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


# ── Image loading ──────────────────────────────────────────────

def load_and_preprocess(path, size=IMG_SIZE):
    if not path or not os.path.exists(path):
        return None
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (size, size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return img


# ── Cropping ──────────────────────────────────────────────────

def crop_box_for_size(w, h):
    x1, y1, x2, y2 = CROP_TOF
    sx, sy = w / TOF_W, h / TOF_H
    return int(x1 * sx), int(y1 * sy), int(x2 * sx), int(y2 * sy)


def crop_all_images(data_dir):
    folders = sorted(
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d)) and d[0].isdigit()
    )
    print(f"[1/3] Cropping images in {len(folders)} folders...")
    total, skipped, corrupt = 0, 0, 0
    for folder in folders:
        folder_path = os.path.join(data_dir, folder)
        for fname in os.listdir(folder_path):
            if not fname.endswith(".jpg") or "_cropped" in fname:
                continue
            if not re.match(r"^pig\d+_", fname):
                continue
            src = os.path.join(folder_path, fname)
            base, ext = os.path.splitext(src)
            dst = base + "_cropped" + ext
            if os.path.exists(dst):
                skipped += 1
                continue
            if os.path.getsize(src) < 1000:
                corrupt += 1
                continue
            img = cv2.imread(src)
            if img is None:
                corrupt += 1
                continue
            h, w = img.shape[:2]
            if h < 10 or w < 10:
                corrupt += 1
                continue
            x1, y1, x2, y2 = crop_box_for_size(w, h)
            cv2.imwrite(dst, img[y1:y2, x1:x2])
            total += 1
    print(f"  Cropped: {total} new, {skipped} already existed, {corrupt} corrupt/skipped")


# ── Depth prefilter ──────────────────────────────────────────

def load_depth_raw(path):
    if not path or not os.path.exists(path):
        return None
    size = os.path.getsize(path)
    expected = TOF_W * TOF_H * 2
    if size == expected + 8:
        raw = np.fromfile(path, dtype=np.uint16, offset=8)
    elif size == expected:
        raw = np.fromfile(path, dtype=np.uint16)
    else:
        return None
    if raw.size != TOF_W * TOF_H:
        return None
    return raw.reshape((TOF_H, TOF_W))


def check_pig_present(depth_raw_path):
    depth = load_depth_raw(depth_raw_path)
    if depth is None:
        return True, 0.0
    x1, y1, x2, y2 = CROP_TOF
    crop = depth[y1:y2, x1:x2].copy()
    crop[:, :BAR_MARGIN] = 0
    crop[:, -BAR_MARGIN:] = 0
    valid = crop[(crop > 200) & (crop < 5000)]
    if len(valid) == 0:
        return False, 0.0
    median = float(np.median(valid))
    return median < DEPTH_THRESHOLD, median


# ── Scan folders ──────────────────────────────────────────────

def scan_folder(data_dir):
    folders = sorted(
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d)) and FOLDER_RE.match(d)
    )
    if not folders:
        return []
    records = {}
    for folder in folders:
        folder_path = os.path.join(data_dir, folder)
        for fname in os.listdir(folder_path):
            m = FILENAME_RE.match(fname)
            if not m:
                continue
            pig_id = int(m.group(1))
            modality = m.group(2)
            timestamp = m.group(3)
            is_cropped = m.group(4) is not None
            key = (folder, pig_id, timestamp)
            if key not in records:
                records[key] = {
                    "timestamp_folder": folder,
                    "pig_id": pig_id,
                    "pig_timestamp": timestamp,
                }
            if modality == "ir_vis":
                mod_key = "ir"
            elif modality == "depth_vis":
                mod_key = "depth"
            else:
                mod_key = modality
            col_name = f"{mod_key}{'_cropped' if is_cropped else ''}_{m.group(5)}"
            records[key][col_name] = os.path.join(folder_path, fname)
    return sorted(records.values(), key=lambda r: (r["timestamp_folder"], r["pig_id"]))


# ── Inference ──────────────────────────────────────────────────

def run_predictions(data_dir, device, model):
    frames = scan_folder(data_dir)
    if not frames:
        print("No frames found.")
        return []
    print(f"[2/3] Running inference on {len(frames)} frames...")
    results = []
    skipped = 0
    skipped_corrupt = 0
    with torch.no_grad():
        for i, frame in enumerate(frames):
            depth_raw = frame.get("depth_raw", "")
            present, median_d = check_pig_present(depth_raw)
            if not present:
                skipped += 1
                continue
            # Prefer cropped IR
            path = ""
            for key in ("ir_cropped_jpg", "ir_jpg"):
                p = frame.get(key, "")
                if p and os.path.exists(p):
                    path = p
                    break
            if not path:
                skipped_corrupt += 1
                continue
            img = load_and_preprocess(path)
            if img is None:
                skipped_corrupt += 1
                continue
            x = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred = int(np.argmax(probs))
            conf = float(probs[pred])
            results.append({
                "timestamp_folder": frame["timestamp_folder"],
                "pig_id": frame["pig_id"],
                "pig_timestamp": frame["pig_timestamp"],
                "prediction": pred,
                "prediction_name": POSTURE3_CLASSES.get(pred, str(pred)),
                "confidence": round(conf, 4),
                "median_depth": round(median_d, 1),
                "image_path": path,
            })
            if (i + 1) % 500 == 0:
                print(f"  {i + 1}/{len(frames)} frames...")
    print(f"  Predicted: {len(results)}, Skipped (empty stall): {skipped}, Skipped (corrupt/missing): {skipped_corrupt}")
    return results


# ── Heatmap ────────────────────────────────────────────────────

def parse_timestamp(ts_str):
    try:
        return datetime.strptime(ts_str, "%Y%m%d-%H-%M-%S")
    except ValueError:
        return None


def generate_heatmap(results, out_path):
    # Normalize pig IDs (camera overflow)
    for r in results:
        r["pig_id"] = r["pig_id"] % 20
    pig_ids = list(range(20))
    timestamps = [parse_timestamp(r["pig_timestamp"]) for r in results]
    valid = [(r, t) for r, t in zip(results, timestamps) if t is not None]
    if not valid:
        print("  No valid timestamps for heatmap")
        return
    min_t = min(t for _, t in valid)
    max_t = max(t for _, t in valid)
    hours = []
    t = min_t.replace(minute=0, second=0)
    while t <= max_t:
        hours.append(t)
        t += timedelta(hours=1)
    if not hours:
        return
    posture_map = {"standing": 0, "sitting": 1, "lying": 2}
    votes = defaultdict(list)
    for r, t in valid:
        hi = int((t - hours[0]).total_seconds() / 3600)
        hi = min(hi, len(hours) - 1)
        votes[(r["pig_id"], hi)].append(posture_map.get(r["prediction_name"], -1))
    pig_idx = {pid: i for i, pid in enumerate(pig_ids)}
    grid = np.full((len(pig_ids), len(hours)), -1, dtype=float)
    for (pid, hi), postures in votes.items():
        if pid in pig_idx:
            grid[pig_idx[pid], hi] = max(set(postures), key=postures.count)
    colors = ["#d4d4d4", "#22c55e", "#f97316", "#3b82f6"]
    cmap = ListedColormap(colors)
    bounds = [-1.5, -0.5, 0.5, 1.5, 2.5]
    norm = BoundaryNorm(bounds, cmap.N)
    fig, ax = plt.subplots(figsize=(max(14, len(hours) * 0.3), 8))
    ax.pcolormesh(grid, cmap=cmap, norm=norm, edgecolors="white", linewidth=0.5)
    ax.set_yticks(np.arange(len(pig_ids)) + 0.5)
    ax.set_yticklabels([f"pig {pid}" for pid in pig_ids], fontsize=8)
    ax.set_ylim(0, len(pig_ids))
    step = max(1, 6)
    ax.set_xticks(np.arange(0, len(hours), step) + 0.5)
    ax.set_xticklabels(
        [hours[i].strftime("%m/%d %H:%M") for i in range(0, len(hours), step)],
        rotation=45, ha="right", fontsize=7,
    )
    ax.set_xlim(0, len(hours))
    ax.set_xlabel("Time")
    ax.set_ylabel("Pig ID")
    ax.set_title("Posture Timeline Heatmap (hourly majority vote)")
    legend_elements = [
        Patch(facecolor="#22c55e", label="Standing"),
        Patch(facecolor="#f97316", label="Sitting"),
        Patch(facecolor="#3b82f6", label="Lying"),
        Patch(facecolor="#d4d4d4", label="No data"),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()
    print(f"  Heatmap saved: {out_path}")


print("Pipeline code loaded.")

## 4. Load Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)
backbone = ckpt.get("backbone", "mobilenet_v2")
model = SingleModalModel(
    in_channels=ckpt.get("in_channels", 3),
    num_classes=ckpt.get("num_classes", 3),
    pretrained=False,
    backbone=backbone,
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Model loaded: {backbone}, {ckpt.get('num_classes', 3)} classes, {sum(p.numel() for p in model.parameters()):,} params")

## 5. Run Pipeline

This will:
1. Crop all images (skip if already cropped)
2. Run posture inference on each frame
3. Save predictions CSV + heatmap PNG

**Estimated time:** ~30 min for 3,766 folders (75K frames) on Colab GPU.

In [ ]:
# Step 1: Crop
crop_all_images(DATA_DIR)

# Step 2: Predict
results = run_predictions(DATA_DIR, device, model)

if results:
    # Step 3: Save CSV
    csv_path = os.path.join(OUTPUT_DIR, "predictions.csv")
    fields = ["timestamp_folder", "pig_id", "pig_timestamp",
              "prediction", "prediction_name", "confidence",
              "median_depth", "image_path"]
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(results)
    print(f"\nSaved predictions: {csv_path}")

    # Step 4: Heatmap
    heatmap_path = os.path.join(OUTPUT_DIR, "posture_heatmap.png")
    generate_heatmap(list(results), heatmap_path)

    # Summary
    counts = Counter(r["prediction_name"] for r in results)
    total = len(results)
    print(f"\n{'='*50}")
    print(f"  Total predicted: {total}")
    for name in ["standing", "sitting", "lying"]:
        c = counts.get(name, 0)
        print(f"  {name:<10}: {c:>6} ({c/total*100:.1f}%)")
    print(f"  Avg confidence: {np.mean([r['confidence'] for r in results]):.3f}")
    print(f"{'='*50}")
else:
    print("No predictions generated. Check your DATA_DIR path.")

## 6. View Results

In [ ]:
# Display the heatmap inline
from IPython.display import Image, display

heatmap_file = os.path.join(OUTPUT_DIR, "posture_heatmap.png")
if os.path.exists(heatmap_file):
    display(Image(filename=heatmap_file))
else:
    print("Heatmap not generated yet. Run the pipeline first.")

In [ ]:
# Preview first 20 rows of predictions
import pandas as pd

csv_file = os.path.join(OUTPUT_DIR, "predictions.csv")
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"Total rows: {len(df)}")
    display(df.head(20))
else:
    print("Predictions CSV not generated yet.")

## 7. Download Results

Run this cell to download the predictions CSV and heatmap to your computer.

In [ ]:
from google.colab import files

csv_file = os.path.join(OUTPUT_DIR, "predictions.csv")
heatmap_file = os.path.join(OUTPUT_DIR, "posture_heatmap.png")

if os.path.exists(csv_file):
    files.download(csv_file)
if os.path.exists(heatmap_file):
    files.download(heatmap_file)